# 🛒 Analyse RFM & Prédiction du Churn — Online Retail II

Ce notebook réalise une analyse complète de la valeur client sur un jeu de données e-commerce.

## Objectifs
1. **Nettoyer** les données transactionnelles brutes
2. **Calculer les scores RFM** (Récence, Fréquence, Montant) par client
3. **Segmenter** les clients en 4 profils (Champion, Fidèle, À risque, Perdu)
4. **Prédire le churn** avec un modèle Random Forest
5. **Stocker et interroger** les résultats en SQLite

## Dataset
**Online Retail II** — transactions d'un détaillant britannique (2009–2011)  
Source : [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Online+Retail+II)

## 1. Chargement et nettoyage des données

On commence par charger le CSV (encodage `latin-1` requis) et supprimer les lignes invalides :
- **Lignes sans `Customer ID`** → transactions anonymes, inutilisables pour l'analyse client
- **Quantité ≤ 0** → retours ou corrections comptables
- **Prix ≤ 0** → erreurs de saisie ou articles offerts
- **Factures commençant par `C`** → avoirs (Cancellations), à exclure du CA réel

In [ ]:
import pandas as pd
import datetime

df = pd.read_csv("online_retail_II.csv", encoding="latin-1")
print(f"Shape brut : {df.shape}")

# Nettoyage : suppression des lignes invalides
df_clean = df.dropna(subset=['Customer ID'])             # Supprime les clients anonymes
df_clean = df_clean[df_clean['Quantity'] > 0]            # Conserve uniquement les ventes positives
df_clean = df_clean[df_clean['Price'] > 0]               # Élimine les prix négatifs ou nuls
df_clean = df_clean[~df_clean['Invoice'].str.startswith('C')]  # Exclut les avoirs

# Typage et création de la colonne TotalPrice
df_clean = df_clean.copy()
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['Price']

print(f"Shape nettoyé : {df_clean.shape}")

## 2. Calcul des métriques RFM

Le modèle **RFM** quantifie le comportement de chaque client sur 3 axes :

| Métrique | Description | Calcul |
|----------|-------------|--------|
| **R**écence | Nombre de jours depuis le dernier achat | `date_ref - date_max_achat` |
| **F**réquence | Nombre de commandes distinctes | `nunique(Invoice)` |
| **M**ontant | Chiffre d'affaires total généré | `sum(Quantity × Price)` |

> **Date de référence** : 10 décembre 2011 (J+10 après la dernière transaction du dataset)

In [ ]:
# Date de référence : légèrement après la dernière date du dataset
reference_date = datetime.datetime(2011, 12, 10)

rfm = df_clean.groupby('Customer ID').agg(
    Recence=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequence=('Invoice', 'nunique'),
    Montant=('TotalPrice', 'sum')
).reset_index()

print(f"Nombre de clients uniques : {rfm.shape[0]}")
print(rfm.head())

## 3. Scoring RFM (quintiles)

Chaque métrique est découpée en **5 quintiles** pour obtenir un score de 1 à 5 :

- **Score R** : inversé → un client récent (faible Récence) reçoit un score **5** (meilleur)
- **Score F & M** : croissant → plus la fréquence/montant est élevé, plus le score est **5**
- `rank(method='first')` évite les doublons dans `pd.qcut` lorsque plusieurs clients ont la même valeur

Le **Score Total** (somme des 3 scores) va de 3 à 15.

In [ ]:
# Score Récence : inversé (petit = récent = bien → score 5)
rfm['Score_R'] = pd.qcut(rfm['Recence'], q=5, labels=[5, 4, 3, 2, 1])

# Score Fréquence : rank() pour éviter les erreurs de doublons dans qcut
rfm['Score_F'] = pd.qcut(rfm['Frequence'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# Score Montant
rfm['Score_M'] = pd.qcut(rfm['Montant'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# Score total (somme des 3 scores)
rfm['Score_Total'] = rfm['Score_R'].astype(int) + rfm['Score_F'].astype(int) + rfm['Score_M'].astype(int)

print(rfm[['Customer ID', 'Score_R', 'Score_F', 'Score_M', 'Score_Total']].head())

## 4. Segmentation des clients

On classe chaque client dans l'un des 4 segments selon son Score Total :

| Segment | Score Total | Interprétation |
|---------|------------|----------------|
| 🏆 Champion | ≥ 13 | Acheteurs récents, fréquents, à forte valeur |
| 💙 Fidèle | 10 – 12 | Clients réguliers à conserver |
| ⚠️ À risque | 7 – 9 | Anciens bons clients qui s'éloignent |
| ❌ Perdu | < 7 | Clients inactifs depuis longtemps |

In [ ]:
def segment_client(score):
    """Attribue un segment RFM à partir du score total (3-15)."""
    if score >= 13:
        return 'Champion'
    elif score >= 10:
        return 'Fidèle'
    elif score >= 7:
        return 'À risque'
    else:
        return 'Perdu'

rfm['Segment'] = rfm['Score_Total'].apply(segment_client)

print(rfm['Segment'].value_counts())
print(rfm.head(10))

## 5. Visualisation — Répartition des segments

Un bar chart simple pour visualiser la distribution des 4 segments.

In [ ]:
import matplotlib.pyplot as plt

segment_counts = rfm['Segment'].value_counts()
colors = ['#27AE60', '#3498DB', '#E67E22', '#E74C3C']

plt.figure(figsize=(8, 5))
bars = plt.bar(segment_counts.index, segment_counts.values, color=colors)
plt.title('Segmentation RFM des clients', fontsize=14, fontweight='bold')
plt.xlabel('Segment')
plt.ylabel('Nombre de clients')

# Étiquettes de valeur au-dessus de chaque barre
for bar, val in zip(bars, segment_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('segmentation_rfm.png', dpi=150)
plt.show()
print("Graphique sauvegardé")

## 6. Tableau de bord RFM (4 graphiques)

Un dashboard complet avec :
- **Camembert** de la répartition des segments
- **Barplot horizontal** du nombre de clients par segment
- **Montant moyen** par segment (valeur économique)
- **Scatter plot** Fréquence vs Montant coloré par segment

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analyse RFM des Clients', fontsize=16, fontweight='bold')

# 1. Camembert — répartition des segments
segment_counts = rfm['Segment'].value_counts()
axes[0, 0].pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%', startangle=140)
axes[0, 0].set_title('Répartition des Segments')

# 2. Nombre de clients par segment
sns.barplot(x=segment_counts.values, y=segment_counts.index, ax=axes[0, 1], palette='viridis')
axes[0, 1].set_title('Nombre de Clients par Segment')
axes[0, 1].set_xlabel('Nombre de clients')

# 3. Montant moyen par segment — indicateur de valeur économique
montant_seg = rfm.groupby('Segment')['Montant'].mean().sort_values(ascending=False)
sns.barplot(x=montant_seg.values, y=montant_seg.index, ax=axes[1, 0], palette='magma')
axes[1, 0].set_title('Montant Moyen par Segment (€)')
axes[1, 0].set_xlabel('Montant moyen (€)')

# 4. Scatter plot Fréquence vs Montant — identifier les outliers par segment
segments_uniques = rfm['Segment'].unique()
couleurs = plt.cm.tab10.colors
for i, seg in enumerate(segments_uniques):
    subset = rfm[rfm['Segment'] == seg]
    axes[1, 1].scatter(subset['Frequence'], subset['Montant'],
                       label=seg, alpha=0.6, color=couleurs[i % len(couleurs)])
axes[1, 1].set_title('Fréquence vs Montant par Segment')
axes[1, 1].set_xlabel('Fréquence')
axes[1, 1].set_ylabel('Montant (€)')
axes[1, 1].legend(fontsize=7)

plt.tight_layout()
plt.savefig('analyse_rfm.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegardé")

## 7. Modèle de prédiction du Churn (Random Forest)

On utilise les métriques RFM comme **features** pour prédire si un client est "Perdu" (churn = 1).

### Pourquoi Random Forest ?
- Robuste aux données non normalisées (pas besoin de scaler les features)
- Fournit naturellement l'importance des variables
- Bonne performance sur données tabulaires déséquilibrées

### Construction de la cible
`Churn = 1` si le segment est `'Perdu'`, sinon `0`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Variable cible binaire : 1 si le client est dans le segment 'Perdu'
rfm['Churn'] = (rfm['Segment'] == 'Perdu').astype(int)

X = rfm[['Recence', 'Frequence', 'Montant']]
y = rfm['Churn']

# Split 80/20 stratifié implicitement (random_state pour reproductibilité)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print(f"Taux de churn : {y.mean()*100:.1f}%")

In [ ]:
# Entraînement du modèle
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Évaluation sur le jeu de test
y_pred = model.predict(X_test)

print("=== PERFORMANCE DU MODÈLE ===")
print(classification_report(y_test, y_pred, target_names=['Non churné', 'Churné']))

### Importance des variables

Le Random Forest calcule l'importance de chaque feature via la réduction moyenne d'impureté de Gini.
On visualise les 3 variables pour comprendre quels signaux prédisent le mieux le churn.

In [ ]:
importances = pd.DataFrame({
    'Feature': ['Recence', 'Frequence', 'Montant'],
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("=== IMPORTANCE DES VARIABLES ===")
print(importances)

plt.figure(figsize=(7, 4))
bars = plt.barh(importances['Feature'], importances['Importance'],
                color=['#E74C3C', '#E67E22', '#3498DB'])
plt.title('Variables les plus prédictives du churn', fontsize=13, fontweight='bold')
plt.xlabel('Importance (Gini)')

# Affichage du pourcentage d'importance
for bar, val in zip(bars, importances['Importance']):
    plt.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.1%}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('importance_variables.png', dpi=150)
plt.show()

## 8. Stockage SQLite et requêtes analytiques

On persiste les données nettoyées et les scores RFM dans une base **SQLite locale** (`retailsense.db`).
Cela permet d'interroger les résultats avec du SQL standard, et de connecter Power BI ou Metabase.

### Tables créées
| Table | Contenu |
|-------|---------|
| `transactions` | Données nettoyées ligne par ligne |
| `clients_rfm` | Scores RFM, segments et flag churn par client |

In [ ]:
import sqlite3

conn = sqlite3.connect('retailsense.db')

# Chargement dans SQLite (replace = recréation complète à chaque run)
df_clean.to_sql('transactions', conn, if_exists='replace', index=False)
rfm.to_sql('clients_rfm', conn, if_exists='replace', index=False)

print("Base de données créée avec succès")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

In [ ]:
# Requête 1 : Top 10 pays par chiffre d'affaires
# → identifier les marchés les plus stratégiques
query1 = """
SELECT Country,
       ROUND(SUM(TotalPrice), 2) as CA_Total,
       COUNT(DISTINCT "Customer ID") as Nb_Clients
FROM transactions
GROUP BY Country
ORDER BY CA_Total DESC
LIMIT 10
"""
top_pays = pd.read_sql(query1, conn)
print("=== TOP 10 PAYS PAR CA ===")
print(top_pays)

In [ ]:
# Requête 2 : Statistiques par segment
# → synthèse pour orienter la stratégie CRM
query2 = """
SELECT Segment,
       COUNT(*) as Nb_Clients,
       ROUND(AVG(Montant), 2) as Montant_Moyen,
       ROUND(AVG(Recence), 1) as Recence_Moyenne
FROM clients_rfm
GROUP BY Segment
ORDER BY Nb_Clients DESC
"""
segments = pd.read_sql(query2, conn)
print("=== ANALYSE PAR SEGMENT ===")
print(segments)

In [ ]:
# Requête 3 : Clients 'À risque' avec montant > 1000€
# → priorité de réactivation CRM (fort potentiel, en train de partir)
query3 = """
SELECT "Customer ID", Recence, Frequence, ROUND(Montant, 2) as Montant, Segment
FROM clients_rfm
WHERE Segment = 'À risque'
AND Montant > 1000
ORDER BY Montant DESC
LIMIT 10
"""
risque_fort = pd.read_sql(query3, conn)
print("=== CLIENTS À RISQUE HAUTE VALEUR ===")
print(risque_fort)

## 9. Export des fichiers

Export des résultats en CSV pour intégration dans Power BI, Tableau ou un rapport GitHub.

| Fichier | Contenu |
|---------|----------|
| `rfm_clients.csv` | Scores RFM complets de tous les clients |
| `top_pays.csv` | Top 10 pays par CA |
| `segments.csv` | Statistiques agrégées par segment |
| `clients_risque_haute_valeur.csv` | Clients prioritaires pour la rétention |

In [ ]:
rfm.to_csv('rfm_clients.csv', index=False)
top_pays.to_csv('top_pays.csv', index=False)
segments.to_csv('segments.csv', index=False)
risque_fort.to_csv('clients_risque_haute_valeur.csv', index=False)

conn.close()

print("Fichiers exportés avec succès :")
print("  - rfm_clients.csv")
print("  - top_pays.csv")
print("  - segments.csv")
print("  - clients_risque_haute_valeur.csv")